PART 04：跑起来——亲眼看着它自己干活、自己修错
光看代码没感觉，跑起来才知道这 142 行有多上头。

python code.py
出现 agent >> 提示符，开测。

实录一：最小任务，看循环转三圈
输入：

帮我在当前目录创建一个 hello.py，内容是打印 Hello, World!，并运行验证
终端上你会看到模型自己开始干活（黄色是它申请的命令）：

$ echo 'print("Hello, World!")' > hello.py
$ python hello.py
Hello, World!
Done. hello.py created and verified.
拆一下账本里发生了什么：

第 1 圈
：模型申请 echo ... > hello.py 写文件 → 结果"(no output)"回流
第 2 圈
：模型申请 python hello.py 验证 → 结果"Hello, World!"回流
第 3 圈
：模型没有再申请任何工具 → 循环退出，打印最终回复
注意一个细节：你只说了"创建"，它自己决定多跑一步验证。没有人教它，流程图里也没有这个节点——这就是 agency 从模型权重里自己长出来的样子。

实录二：名场面——跑挂了，它自己修
来个有难度的。输入：

写一个 divide.py，计算 10 除以 0 并打印结果，然后运行它
模型会老老实实写文件、运行，然后：

$ python divide.py
Traceback (most recent call last):
  File "divide.py", line 1, in <module>
    print(10 / 0)
ZeroDivisionError: division by zero
红色的 Traceback 刷出来。接下来发生的事，是这个内核最接近"生命"的一刻——它没有停下来问你怎么办，而是：

$ cat divide.py
$ echo 'try:
    print(10 / 0)
except ZeroDivisionError:
    print("不能除以零")' > divide.py
$ python divide.py
不能除以零
Fixed: added exception handling.
看明白发生了什么吗？

Traceback 不是给你看的，是给模型看的。

stderr 被原样塞进了 tool_result，下一圈循环，模型读到自己的报错，自己定位、自己改、自己再跑。错误信息就是它的眼睛——这就是"观察-行动闭环"：观察（读报错）→ 行动（改代码）→ 再观察（看新结果），直到满意为止。

你平时用 Claude Code 时它"改 bug 改到通为止"的体感，底层就是这个循环在转。

实录三：反例——它不是什么都要动手
最后问个不用动手的：

1+1 等于几？
模型直接回复"2"，从头到尾一个工具都没调，循环转一圈就退出。

呼应机制三：调不调工具、什么时候停，全在模型的一念之间。该动手时动手，不该动手时闭嘴直接答——这个分寸感，同样来自模型，不来自你的代码。

一句严肃的安全提醒
再次强调：这段代码会真实执行模型生成的 shell 命令。黑名单只挡得住四条最狠的。请在一个专门的临时目录里玩，别对着有生产代码、有重要文件的目录跑。像样的权限门禁，是后面要单独拆的机制。